In [3]:
import requests
import time
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
import json

FASTAPI_URL = "http://fastapi_server:8000/process-meeting/"
TEST_AUDIO_PATH = "/home/jovyan/work/test_audio.mp3" 

NUM_CONCURRENT_USERS = 10  # Simulate 10 users sending requests at once
NUM_REQUESTS_PER_USER = 10 # Each simulated user sends this many requests
REQUEST_TIMEOUT_SECONDS = 60 # Timeout for each request

# --- Helper to send a single request ---
def send_processing_request(url, audio_file_path, query_text=None):
    try:
        with open(audio_file_path, 'rb') as f_audio:
            files = {'audio_file': (os.path.basename(audio_file_path), f_audio, 'audio/wav')}
            data = {}
            if query_text is not None:
                data['query'] = query_text

            print(f"Sending request to {url}...")
            # Send the POST request
            response = requests.post(url, files=files, data=data, timeout=REQUEST_TIMEOUT_SECONDS)
            print(f"Received response: Status {response.status_code}")

            # Optional: Process response if needed
            try:
                response_json = response.json()
                print(f"Response JSON: {response_json}")
            except json.JSONDecodeError:
                print(f"Could not decode JSON response: {response.text}")

            return response.status_code

    except FileNotFoundError:
        print(f"Error: Test audio file not found at {audio_file_path}")
        return "File Not Found"
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return f"Request Error: {e}"
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return f"Unexpected Error: {e}"

# --- Main Load Test Execution ---
if __name__ == "__main__":
    if not os.path.exists(TEST_AUDIO_PATH):
        print(f"ERROR: Test audio file not found at {TEST_AUDIO_PATH}. Please provide a valid audio file.")
        exit(1)

    total_requests_to_send = NUM_CONCURRENT_USERS * NUM_REQUESTS_PER_USER
    print(f"Starting load test:")
    print(f"  URL: {FASTAPI_URL}")
    print(f"  Test Audio: {TEST_AUDIO_PATH}")
    print(f"  Concurrent Users: {NUM_CONCURRENT_USERS}")
    print(f"  Requests per User: {NUM_REQUESTS_PER_USER}")
    print(f"  Total Requests: {total_requests_to_send}")
    print("-" * 20)

    start_time = time.time()
    results = [] # Store status codes or error indicators

    with ThreadPoolExecutor(max_workers=NUM_CONCURRENT_USERS) as executor:
        # Submit tasks for each request
        futures = []
        for i in range(NUM_CONCURRENT_USERS):
            for j in range(NUM_REQUESTS_PER_USER):
                 # Example: Send some summarization (query=None) and some Q&A requests
                query = "What are the main points?" if (i*NUM_REQUESTS_PER_USER + j) % 2 == 0 else None # Alternate request types
                future = executor.submit(send_processing_request, FASTAPI_URL, TEST_AUDIO_PATH, query)
                futures.append(future)

        # Wait for all futures to complete and collect results
        for future in as_completed(futures):
            results.append(future.result())

    end_time = time.time()
    duration = end_time - start_time

    # --- Summarize Results ---
    successful_count = sum(1 for r in results if isinstance(r, int) and 200 <= r < 300)
    client_error_count = sum(1 for r in results if isinstance(r, int) and 400 <= r < 500)
    server_error_count = sum(1 for r in results if isinstance(r, int) and 500 <= r < 600)
    request_failed_count = sum(1 for r in results if isinstance(r, str) and r.startswith("Request Error:"))
    file_not_found_count = sum(1 for r in results if r == "File Not Found")
    unexpected_error_count = sum(1 for r in results if isinstance(r, str) and r.startswith("Unexpected Error:"))

    total_processed = len(results) # Number of requests that completed (successfully or with error/failure)

    print("-" * 20)
    print(f"Load test completed in {duration:.2f} seconds")
    print(f"Total requests initiated: {total_requests_to_send}")
    print(f"Total requests processed: {total_processed} (includes successes, errors, and failures)")
    print(f"  Successful (2xx): {successful_count}")
    print(f"  Client Errors (4xx): {client_error_count}")
    print(f"  Server Errors (5xx): {server_error_count}")
    print(f"  Request Failures (e.g., timeout, connection): {request_failed_count}")
    print(f"  File Not Found Errors: {file_not_found_count}")
    print(f"  Unexpected Errors: {unexpected_error_count}")


    if duration > 0:
         print(f"\nOverall Request Rate: {total_processed / duration:.2f} requests/sec")
    else:
         print("\nOverall Request Rate: N/A (duration was 0)")

Starting load test:
  URL: http://fastapi_server:8000/process-meeting/
  Test Audio: /home/jovyan/work/test_audio.mp3
  Concurrent Users: 10
  Requests per User: 10
  Total Requests: 100
--------------------
Sending request to http://fastapi_server:8000/process-meeting/...
Sending request to http://fastapi_server:8000/process-meeting/...
Sending request to http://fastapi_server:8000/process-meeting/...
Sending request to http://fastapi_server:8000/process-meeting/...
Sending request to http://fastapi_server:8000/process-meeting/...
Sending request to http://fastapi_server:8000/process-meeting/...
Request failed: HTTPConnectionPool(host='fastapi_server', port=8000): Max retries exceeded with url: /process-meeting/ (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x78fb6ab7cd70>: Failed to resolve 'fastapi_server' ([Errno -3] Temporary failure in name resolution)"))
Request failed: HTTPConnectionPool(host='fastapi_server', port=8000): Max retries exceeded with